# 2.0.0 Seed semantic classification

This notebook builds the retained seed-label set for the semantic classifier. It samples from the annotation-support corpus (*shadow corpus*) rather than the main analysis corpus, uses an OpenAI model for the first-pass labeling campaign, and then runs local reliability checks before the larger seed pass is accepted.

Operationally, **this is a high-cost stage** and should be treated as a retained labeling campaign rather than a routine rerun surface. Its job is to produce the seed labels and audit summaries that the later classifier notebooks inherit.

The default configuration is the cached reproduction path: `RUN_TAG = "full_run"` together with `RUN_MODE = "replay"`. That setting *reuses the retained full annotation cache* that ships with the repo, avoids new OpenAI API calls, and lets readers continue through the downstream notebook sequence even if they do not want to rerun the paid GPT labeling stage. If you **do** want to rerun this stage interactively, switch to `RUN_TAG = "run_1"` or `"run_2"` and use `RUN_MODE = "live"` or `"resume"` as appropriate for the 200-item reliability workflow.


### Main variables to decide on:

- RUN_ID: which corpus run is being labeled
- API/model settings for the GPT-assisted annotation stage
- Train/eval split choices and fine-tuning hyperparameters for the SciBERT classifier
- Category thresholds used when converting classifier probabilities into final categorical labels
- Any file paths controlling where sampled annotation batches, trained checkpoints, and relabeled corpora are written

### Outputs

Depending on the section being run, this notebook writes annotated seed samples, reliability summaries, fine-tuned classifier checkpoints, and corpus-level classification outputs. Those outputs provide the label assignments used by the empirical and significance notebooks.

---

## Initialize the run

---

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def _find_workspace_root() -> Path:
    env_root = os.environ.get("WORKSPACE_ROOT", "").strip()
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "config" / "workspace.json").exists():
            return candidate
    for start in [Path.cwd(), *Path.cwd().parents]:
        if (start / "config" / "workspace.json").exists():
            return start
    raise FileNotFoundError(
        "Could not locate workspace root from WORKSPACE_ROOT or the current working directory."
    )


WORKSPACE_DIR = _find_workspace_root()
PAPER_ROOT = WORKSPACE_DIR.parent
RESEARCH_ROOT = PAPER_ROOT.parent
SHARED_ASSETS_DIR = RESEARCH_ROOT / "shared-assets"
SHARED_CODE_DIR = SHARED_ASSETS_DIR / "code"
if str(SHARED_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_CODE_DIR))

CODE_DIR = WORKSPACE_DIR / "code"
NOTEBOOKS_DIR = CODE_DIR / "notebooks"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from workspace_rooting.workspace_paths import canonical_workspace_paths

paths = canonical_workspace_paths(WORKSPACE_DIR)
WORKSPACE_DIR = paths["workspace"]
CODE_DIR = paths["code"]
CONFIG_DIR = paths["config"]
DATA_DIR = paths["data"]
OUTPUTS_DIR = paths["outputs"]
DOCS_DIR = paths["docs"]
LOCAL_DIR = paths["local"]
SHARED_ASSETS_DIR = paths["shared_assets"]
NOTEBOOKS_DIR = CODE_DIR / "notebooks"

print("workspace_root:", WORKSPACE_DIR)
print("shared_assets_root:", SHARED_ASSETS_DIR)

import numpy as np
import json
import csv
import ast
import random
from pathlib import Path
from types import SimpleNamespace
import os
from dotenv import load_dotenv
import time
from collections import Counter, defaultdict
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
from sklearn.metrics import cohen_kappa_score

load_dotenv()

RUN_ID = "analysis_corpus"
SHADOW_RUN_ID = "shadow_corpus"
SHADOW_SET_ID = "shadow_balanced_1800"

PATHS = SimpleNamespace(
    shadow_source=LOCAL_DIR / "corpora" / SHADOW_RUN_ID / "annotation-sets" / SHADOW_SET_ID / f"{SHADOW_SET_ID}.jsonl",
    gpt_runs=OUTPUTS_DIR / "analytical-results" / "models" / "gpt" / "runs",
)
PATHS.gpt_runs.mkdir(parents=True, exist_ok=True)

print("shadow_source:", PATHS.shadow_source, "| exists:", PATHS.shadow_source.exists())
print("gpt_runs:", PATHS.gpt_runs)

from run_modes.api_run_mode import (
    RunMode,
    append_jsonl_row,
    load_jsonl_rows,
    rebuild_or_resume_message,
    require_env_var,
    resolve_run_mode,
    should_create_client,
)


## GPT-assisted labeling run

---

### GPT configuration


In [ ]:
# ── Config ───────────────────────────────────────────────────────
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")


RUN_MODE = "replay"   # default cached repo-friendly path; use "live" or "resume" only when rerunning annotation
RUN_TAG = "full_run"  # default cached downstream-compatible path; use run_1/run_2 for the 200-item reliability workflow

PROMPT_ID = "pmpt_69b9796f2e388197ba68d1ea61e8935d0468c0362583c5ad"
PROMPT_VERSION = "5"
EXPECTED_MODEL = "gpt-5.4-2026-03-05"
USE_HIGHLIGHTED_SENTENCE = True
STRICT_MODEL_CHECK = True

RESULTS_RUN_ID = f"{SHADOW_SET_ID}__gpt54_prompt_v{PROMPT_VERSION}"
RESULTS_DIR = PATHS.gpt_runs / RESULTS_RUN_ID
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IN_PATH = PATHS.shadow_source
SAMPLE_PATH = RESULTS_DIR / "reliability_sample_ids.json"
RUN_CSV_PATH = RESULTS_DIR / "shadow_gpt_run.csv"
RUN_MANIFEST_PATH = RESULTS_DIR / "run_manifest.json"

if RUN_TAG == "full_run":
    with open(IN_PATH, "r", encoding="utf-8") as f:
        N_TEST = sum(1 for _ in f)
else:
    N_TEST = 200

OUT_PATH = RESULTS_DIR / f"annotation_labeled_{N_TEST}_{RUN_TAG}.jsonl"
SUMMARY_PATH = RESULTS_DIR / f"annotation_results_{N_TEST}_{RUN_TAG}_summary.csv"

SEED = 42
RNG = np.random.default_rng(SEED)

MAX_RETRIES = 5
BASE_SLEEP = 0.7
REQUEST_SLEEP = 0.0

print("RESULTS_DIR:", RESULTS_DIR)
print("IN_PATH:", IN_PATH)
print("OUT_PATH:", OUT_PATH)
print("SUMMARY_PATH:", SUMMARY_PATH)
print("RUN_CSV_PATH:", RUN_CSV_PATH)
print("RUN_MANIFEST_PATH:", RUN_MANIFEST_PATH)
print("PROMPT_ID:", PROMPT_ID)
print("PROMPT_VERSION:", PROMPT_VERSION)
print("EXPECTED_MODEL:", EXPECTED_MODEL)
print("RUN_MODE:", RUN_MODE)
print("STRICT_MODEL_CHECK:", STRICT_MODEL_CHECK)


### Functions for GPT-assisted annotation and reliability checks


In [ ]:
def load_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def append_jsonl(path: Path, row: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def stratified_sample(records, n: int, key="bin", rng=RNG):
    groups = defaultdict(list)
    for r in records:
        groups[r.get(key, "NA")].append(r)

    keys = list(groups.keys())
    sizes = np.array([len(groups[k]) for k in keys], dtype=float)
    if sizes.sum() == 0:
        return []

    props = sizes / sizes.sum()
    target = np.floor(props * n).astype(int)

    remaining = n - target.sum()
    order = np.argsort(-sizes)
    i = 0
    while remaining > 0:
        target[order[i % len(order)]] += 1
        remaining -= 1
        i += 1

    out = []
    for k, t in zip(keys, target):
        g = groups[k]
        if t <= 0:
            continue
        t = min(t, len(g))
        idx = rng.choice(len(g), size=t, replace=False)
        out.extend([g[i] for i in idx])

    if len(out) > n:
        idx = rng.choice(len(out), size=n, replace=False)
        out = [out[i] for i in idx]

    return out


def extract_response_text(response):
    text = getattr(response, "output_text", None)
    if text:
        return text

    chunks = []
    for item in getattr(response, "output", []) or []:
        for content in getattr(item, "content", []) or []:
            text_value = getattr(content, "text", None)
            if text_value:
                chunks.append(text_value)

    if chunks:
        return "".join(chunks)

    raise ValueError("Response did not include text output.")


def resolve_prompt_sentence(row):
    sentence = row.get("sentence_marked") if USE_HIGHLIGHTED_SENTENCE else row.get("sentence") or row.get("sent")
    if not sentence:
        raise ValueError(f"Missing prompt sentence for row {row.get('id')}")
    return sentence


def call_saved_prompt_annotation(client: OpenAI, row):
    sentence = resolve_prompt_sentence(row)
    last_err = None

    for attempt in range(MAX_RETRIES):
        try:
            response = client.responses.create(
                prompt={
                    "id": PROMPT_ID,
                    "version": PROMPT_VERSION,
                    "variables": {
                        "id": row["id"],
                        "sentence_marked": sentence,
                    },
                }
            )

            text = extract_response_text(response).strip()
            data = json.loads(text)
            if not isinstance(data, dict):
                raise ValueError(f"Expected dict JSON response, got {type(data).__name__}")

            model_name = getattr(response, "model", None)
            if STRICT_MODEL_CHECK and model_name and model_name != EXPECTED_MODEL:
                raise RuntimeError(f"Model mismatch: expected {EXPECTED_MODEL}, got {model_name}")

            return {
                "labels": data.get("labels", []),
                "confidence": data["confidence"],
                "note": data.get("note", ""),
                "model": model_name,
                "response_id": getattr(response, "id", None),
            }
        except Exception as e:
            last_err = str(e)
            time.sleep(BASE_SLEEP * (2 ** attempt))

    raise RuntimeError(f"Saved-prompt annotation failed after {MAX_RETRIES} retries: {last_err}")


def filter_by_ids(records, ids):
    idset = set(ids)
    return [r for r in records if r.get("id") in idset]


def load_existing_rows(path: Path, valid_ids=None):
    if not path.exists():
        return []

    valid_ids = set(valid_ids or [])
    rows = []
    seen = set()
    dropped = 0
    for row in load_jsonl(path):
        row_id = row.get("id")
        if valid_ids and row_id not in valid_ids:
            dropped += 1
            continue
        if row_id in seen:
            continue
        seen.add(row_id)
        rows.append(row)

    if dropped:
        print(f"Dropped {dropped} checkpoint rows not present in this sample.")
    return rows


def refresh_run_artifacts(rows, sample):
    sample_order = {r["id"]: i for i, r in enumerate(sample)}
    rows = sorted(rows, key=lambda r: sample_order.get(r["id"], 10**9))

    df = pd.DataFrame(rows)
    df["labels_str"] = df["labels"].apply(lambda xs: "+".join(sorted(xs)) if isinstance(xs, list) else "")
    summary = (
        df.groupby(["bin", "labels_str", "confidence"], as_index=False)
          .size()
          .sort_values(["bin", "size"], ascending=[True, False])
    )

    summary.to_csv(SUMMARY_PATH, index=False)
    df.to_csv(RUN_CSV_PATH, index=False)

    run_manifest = {
        "results_run_id": RESULTS_RUN_ID,
        "SHADOW_SET_ID": SHADOW_SET_ID,
        "run_tag": RUN_TAG,
        "prompt_id": PROMPT_ID,
        "prompt_version": PROMPT_VERSION,
        "expected_model": EXPECTED_MODEL,
        "input_path": str(IN_PATH),
        "output_path": str(OUT_PATH),
        "summary_path": str(SUMMARY_PATH),
        "csv_path": str(RUN_CSV_PATH),
        "n_rows": len(rows),
        "bin_counts": dict(Counter(r["bin"] for r in rows)),
    }
    RUN_MANIFEST_PATH.write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
    return summary, rows


## Main annotation loop

---

In [ ]:
def rebuild_cached_artifacts(sample, out_path, summary_path, run_csv_path, run_manifest_path):
    out_rows = load_jsonl_rows(out_path)
    if not out_rows:
        raise FileNotFoundError(f"No cached run found at {out_path}")

    summary, out_rows = refresh_run_artifacts(out_rows, sample)

    print("Rebuilt from cache:", out_path)
    print("Wrote:", summary_path)
    print("Wrote:", run_csv_path)
    print("Wrote:", run_manifest_path)
    return summary, out_rows


In [ ]:
records = list(load_jsonl(IN_PATH))

# ── Input conditional on run option ───────────────────────────────────────────────────────
if RUN_TAG == "run_1":
    sample = stratified_sample(records, N_TEST, key="bin", rng=RNG)
    sample_ids = [r["id"] for r in sample]
    SAMPLE_PATH.write_text(json.dumps(sample_ids, indent=2), encoding="utf-8")
    print("Saved reliability sample IDs:", SAMPLE_PATH)
elif RUN_TAG == "run_2":
    sample_ids = json.loads(SAMPLE_PATH.read_text(encoding="utf-8"))
    sample = filter_by_ids(records, sample_ids)
    id_to_r = {r["id"]: r for r in sample}
    sample = [id_to_r[i] for i in sample_ids if i in id_to_r]
    print("Loaded reliability sample IDs:", SAMPLE_PATH)
elif RUN_TAG == "full_run":
    sample = list(records)
    print("Using the full designated annotation set without resampling.")
else:
    raise ValueError(f"Unknown RUN_TAG: {RUN_TAG}")

sample_ids = [r["id"] for r in sample]
print("sample size:", len(sample))
print("bin counts:", Counter(r["bin"] for r in sample))

state = resolve_run_mode(
    run_mode=RUN_MODE,
    cache_path=OUT_PATH,
    expected_ids=sample_ids,
    completed_id_fn=lambda rows: {r["id"] for r in rows},
)
print(rebuild_or_resume_message(state))

if state.mode is RunMode.REPLAY:
    summary, out_rows = rebuild_cached_artifacts(
        sample,
        OUT_PATH,
        SUMMARY_PATH,
        RUN_CSV_PATH,
        RUN_MANIFEST_PATH,
    )
    print(summary.head(30))

elif state.mode in {RunMode.LIVE, RunMode.RESUME}:
    require_env_var("OPENAI_API_KEY")
    client = OpenAI() if should_create_client(state) else None

    out_rows = load_jsonl_rows(OUT_PATH)
    pending_rows = [r for r in sample if r["id"] in set(state.pending_ids)]

    for row in tqdm(pending_rows, total=len(sample), initial=state.completed_n, desc="Annotating with saved prompt"):
        ann = call_saved_prompt_annotation(client, row)

        out = dict(row)
        out["labels"] = ann["labels"]
        out["confidence"] = ann["confidence"]
        out["note"] = ann.get("note", "")
        out["model"] = ann.get("model") or EXPECTED_MODEL
        out["response_id"] = ann.get("response_id")
        out["prompt_id"] = PROMPT_ID
        out["prompt_version"] = PROMPT_VERSION
        out["sentence_field"] = "sentence_marked" if USE_HIGHLIGHTED_SENTENCE else "sentence"
        out["run_tag"] = RUN_TAG
        out["results_run_id"] = RESULTS_RUN_ID

        append_jsonl_row(OUT_PATH, out)
        out_rows.append(out)

        if REQUEST_SLEEP > 0:
            time.sleep(REQUEST_SLEEP)

    summary, out_rows = refresh_run_artifacts(out_rows, sample)

    print("Wrote checkpoint:", OUT_PATH)
    print("Wrote:", SUMMARY_PATH)
    print("Wrote:", RUN_CSV_PATH)
    print("Wrote:", RUN_MANIFEST_PATH)
    print(summary.head(30))

else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")


## Summary statistics and inspection of results

---

In [ ]:
# ── Total amounts ───────────────────────────────────────────────────────
n_total = summary["size"].sum()
print("Total:", n_total)

# Count how many rows include each label
label_prevalence = {}
for label in ["A", "B", "C"]:
    count = summary.loc[
        summary["labels_str"].fillna("").str.contains(label, regex=False),
        "size"
    ].sum()
    label_prevalence[label] = {
        "count": int(count),
        "prevalence": float(count / n_total)
    }

print(label_prevalence)


# ── Distribution of classes ───────────────────────────────────────────────────────
labelset_dist = (
    summary.groupby("labels_str", as_index=False)["size"]
    .sum()
    .sort_values("size", ascending=False)
)

labelset_dist["share"] = labelset_dist["size"] / labelset_dist["size"].sum()
print(labelset_dist)


In [ ]:
def load_ann(path: Path):
    d = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            d[r["id"]] = r
    return d

def labelset_str(labels):
    if not labels:
        return ""
    return "".join(sorted(set(labels)))

def jaccard(a, b):
    A = set(a or [])
    B = set(b or [])
    if not A and not B:
        return 1.0
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

def reliability_report(path1: Path, path2: Path, cats=("A", "B", "C")):
    r1 = load_ann(path1)
    r2 = load_ann(path2)

    common = sorted(set(r1.keys()) & set(r2.keys()))
    if not common:
        raise ValueError("No overlapping IDs between runs.")

    y1 = [labelset_str(r1[i].get("labels", [])) for i in common]
    y2 = [labelset_str(r2[i].get("labels", [])) for i in common]

    exact = np.mean([a == b for a, b in zip(y1, y2)])
    kappa_exact = cohen_kappa_score(y1, y2)

    per = []
    for c in cats:
        b1 = np.array([1 if c in (r1[i].get("labels", []) or []) else 0 for i in common], dtype=int)
        b2 = np.array([1 if c in (r2[i].get("labels", []) or []) else 0 for i in common], dtype=int)
        k = cohen_kappa_score(b1, b2)
        po = float(np.mean(b1 == b2))
        prev1 = float(b1.mean())
        prev2 = float(b2.mean())
        per.append({
            "label": c,
            "kappa": float(k),
            "agree": po,
            "prev_run1": prev1,
            "prev_run2": prev2,
        })

    jac = float(np.mean([jaccard(r1[i].get("labels", []), r2[i].get("labels", [])) for i in common]))

    return {
        "n_items": len(common),
        "exact_match_rate": float(exact),
        "exact_match_kappa": float(kappa_exact),
        "mean_jaccard": jac,
        "per_label": per,
    }

if N_TEST == 200:
    RUN1_PATH = RESULTS_DIR / f"annotation_labeled_{N_TEST}_run_1.jsonl"
    RUN2_PATH = RESULTS_DIR / f"annotation_labeled_{N_TEST}_run_2.jsonl"

    missing_paths = [path for path in (RUN1_PATH, RUN2_PATH) if not path.exists()]
    if missing_paths:
        print("Reliability report is pending the second adjudication run.")
        print("Missing file(s):")
        for path in missing_paths:
            print(" -", path.name)
    else:
        rep = reliability_report(RUN1_PATH, RUN2_PATH)

        print("=== Reliability Report ===")
        print("Run 1:", RUN1_PATH.name)
        print("Run 2:", RUN2_PATH.name)
        print("N overlapping samples:", rep["n_items"])
        print("Exact match %:", round(rep["exact_match_rate"], 5))
        print("Exact-match kappa:", round(rep["exact_match_kappa"], 5))
        print("Mean Jaccard:", round(rep["mean_jaccard"], 5))

        print()
        print("Per-label kappas:")
        for row in rep["per_label"]:
            print(row)

        reliability_summary_path = RESULTS_DIR / f"reliability_summary_{N_TEST}.json"
        with open(reliability_summary_path, "w", encoding="utf-8") as f:
            json.dump(rep, f, indent=2)

        print()
        print(f"Saved reliability report -> {reliability_summary_path}")
        summary_row = pd.DataFrame([{
            "run_1": RUN1_PATH.name,
            "run_2": RUN2_PATH.name,
            "n_items": rep["n_items"],
            "exact_match_rate": rep["exact_match_rate"],
            "exact_match_kappa": rep["exact_match_kappa"],
            "mean_jaccard": rep["mean_jaccard"],
        }])

        summary_csv_path = RESULTS_DIR / f"reliability_summary_{N_TEST}.csv"
        summary_row.to_csv(summary_csv_path, index=False)

        per_label_csv_path = RESULTS_DIR / f"reliability_summary_{N_TEST}_per_label.csv"
        pd.DataFrame(rep["per_label"]).to_csv(per_label_csv_path, index=False)

        print(f"Saved reliability summary CSV -> {summary_csv_path}")
        print(f"Saved per-label reliability CSV -> {per_label_csv_path}")
else:
    print("Reliability report is only defined for the 200-item run_1/run_2 overlap sample.")
